# FD002 - Robustez del candidato final

Objetivo: revisar si habia una mejora nueva suficientemente interesante sobre el modelo FD002 ya seleccionado. La prueba compara los candidatos fuertes en varios splits artificiales por unidad. Si la mejora no es clara, no se cambia el modelo final.


## Criterio de decision

El modelo vigente `xgb_condition_fault_sensitive_mid_guard` habia mejorado el C-MAPSS del split 42 frente al XGB condition-normalized anterior. Para evitar elegir por un unico split, aca se mira robustez multi-split. La decision queda conservadora: mantener el modelo actual si no aparece una alternativa con mejora clara y estable en C-MAPSS, RMSE y riesgo de errores peligrosos.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parents[2] if Path.cwd().name == 'modeling' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / 'results' / 'FD002'
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'FD002' / 'fd002_best_model_config.json'

robust_summary = pd.read_csv(RESULTS_DIR / 'fd002_robustness_probe_summary.csv')
robust_detail = pd.read_csv(RESULTS_DIR / 'fd002_robustness_probe_detail.csv')
ranking = pd.read_csv(RESULTS_DIR / 'fd002_final_candidate_ranking.csv')
validation_metrics = pd.read_csv(RESULTS_DIR / 'fd002_best_validation_metrics.csv')
official_metrics = pd.read_csv(RESULTS_DIR / 'fd002_official_test_metrics.csv')


## Resultado multi-split

La mejora del candidato vigente existe, pero es chica cuando se promedia en splits. Tambien aparece un trade-off: gana apenas en C-MAPSS promedio, pero no en RMSE promedio ni en worst split.


In [ ]:
robust_summary.style.format({
    'mean_cmapss': '{:.3f}',
    'std_cmapss': '{:.3f}',
    'worst_cmapss': '{:.3f}',
    'mean_rmse': '{:.3f}',
    'mean_dangerous': '{:.2f}',
})


In [ ]:
cols = ['random_state', 'candidate_label', 'cmapss_score', 'rmse', 'dangerous_error_pct']
robust_detail[cols].sort_values(['random_state', 'cmapss_score']).style.format({
    'cmapss_score': '{:.3f}',
    'rmse': '{:.3f}',
    'dangerous_error_pct': '{:.2f}',
})


## Comparacion con el ranking original

En el split 42 el candidato `condition_fault_sensitive_mid_guard` era el mejor por C-MAPSS. La prueba de robustez muestra que esa decision es razonable, pero no abre una nueva mejora. Por eso no se aplica ningun cambio extra a `src/fd002_modeling.py`.


In [ ]:
ranking.head(8)[[
    'experiment_stage', 'candidate_label', 'model_type', 'feature_set',
    'window_size', 'sample_weight_scheme', 'cmapss_score', 'rmse', 'dangerous_error_pct'
]].style.format({
    'cmapss_score': '{:.3f}',
    'rmse': '{:.3f}',
    'dangerous_error_pct': '{:.2f}',
})


## Metricas finales vigentes

Estas metricas son las del modelo FD002 vigente. El test oficial se reporta despues de la seleccion y no se usa para elegir hiperparametros.


In [ ]:
display(validation_metrics)
display(official_metrics)


## Regeneracion opcional

La celda siguiente rerunnea la prueba multi-split. Puede tardar bastante porque reentrena los cuatro candidatos fuertes en cinco splits.


In [ ]:
# Regeneracion opcional, no necesaria para leer el notebook.
# from src.fd002_modeling import evaluate_configs, config_from_metric_row
# labels = [
#     'xgb_condition_fault_sensitive_mid_guard',
#     'xgb_condition_fault_sensitive_weighted',
#     'xgb_squarederror_condition_normalized_weighted',
#     'lgbm_quantile_a04_condition_normalized_weighted',
# ]
# configs = [config_from_metric_row(ranking.loc[ranking['candidate_label'].eq(label)].iloc[0]) for label in labels]
# rows = []
# for state in [0, 1, 2, 3, 4]:
#     metrics, _, _ = evaluate_configs(configs, random_state=state)
#     metrics['eval_random_state'] = state
#     rows.append(metrics)
# detail = pd.concat(rows, ignore_index=True)
# summary = (detail.groupby('candidate_label')
#     .agg(mean_cmapss=('cmapss_score', 'mean'), std_cmapss=('cmapss_score', 'std'),
#          worst_cmapss=('cmapss_score', 'max'), mean_rmse=('rmse', 'mean'),
#          mean_dangerous=('dangerous_error_pct', 'mean'))
#     .reset_index()
#     .sort_values(['mean_cmapss', 'mean_dangerous', 'mean_rmse']))
# detail.to_csv(RESULTS_DIR / 'fd002_robustness_probe_detail.csv', index=False)
# summary.to_csv(RESULTS_DIR / 'fd002_robustness_probe_summary.csv', index=False)
# summary


## Conclusion

No se encontro una mejora nueva que valga la pena aplicar. El modelo vigente se mantiene: `xgb_condition_fault_sensitive_mid_guard`. La lectura importante es que FD002 ya estaba en una zona de trade-off: el agregado fault-sensitive mejora C-MAPSS en promedio por poco, pero no domina todas las metricas de robustez.
